# 2. Preprocesado
Transformación del dataset y creación de preprocesadores para aplicar modelos de clasificación y regresión.

### Imports y carga de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

df = pd.read_csv('../rendimiento_estudiantes.csv', sep=';')

os.makedirs('../preprocessors/', exist_ok=True)

### Valores nulos y duplicados
Primero, se comprueba si hay valores nulos o duplicados y en caso de que haya, se eliminan.

In [2]:
nulos = df.isnull().sum()
print('Valores nulos por variable:')
print(nulos[nulos > 0] if nulos.any() else 'No hay valores nulos en el dataset.')

Valores nulos por variable:
No hay valores nulos en el dataset.


> Esto ya se vio en el apartado de EDA, pero viene bien comprobarlo antes de procesar el dataset.

In [3]:
n_dup = df.duplicated().sum()
print(f'Filas duplicadas: {n_dup}')
if n_dup > 0:
    df = df.drop_duplicates()
    print(f'Eliminadas. Nuevo shape: {df.shape}')

Filas duplicadas: 0


### Outliers
Se detectan outliers mediante el rango intercuartilico (IQR). No se eliminan, ya que los outliers pueden ser casos reales con situaciones extremas. Más adelante se usa StandardScaler, que escala las variables numéricas a media 0 y desviación típica 1.

In [4]:
num_cols = df.select_dtypes(include='number').columns.tolist()
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1
outliers = ((df[num_cols] < Q1 - 1.5*IQR) | (df[num_cols] > Q3 + 1.5*IQR)).sum()
print('Outliers por variable:')
print(outliers[outliers > 0])

Outliers por variable:
orden_solicitud                    541
nota_cualificacion_previa          179
nota_admision                       86
edad_al_matricularse               441
asignaturas_1sem_convalidadas      577
asignaturas_1sem_matriculadas      424
asignaturas_1sem_evaluadas         158
asignaturas_1sem_aprobadas         180
nota_media_1sem                    726
asignaturas_1sem_sin_evaluacion    294
asignaturas_2sem_convalidadas      530
asignaturas_2sem_matriculadas      369
asignaturas_2sem_evaluadas         109
asignaturas_2sem_aprobadas          44
nota_media_2sem                    877
asignaturas_2sem_sin_evaluacion    282
dtype: int64


### Variables excluidas por _data leakage_
Para la tarea de regresión, hay que excluir ciertas variables ya que tienen relación con la variable objetivo. Como se quiere predecir la nota del segundo semestre, todas las variables que tengan que ver con ese periodo (asignaturas aprobadas, asignaturas matriculadas, etc.) se eliminan del dataset para que no aporten información que no estaría disponible en el momento de la predicción.

In [5]:
leak_cols = [c for c in df.columns if '2sem' in c]  # Sin 'objetivo' porque es categórica, pero también se excluiría
print('Variables excluidas en regresión (leakage):')
for col in leak_cols:
    print(f' - {col}')
print(' - objetivo')

# Correlación de las variables de leakage con el target de regresión
corr_leak = df[leak_cols].corr()['nota_media_2sem'].drop('nota_media_2sem')
print('\nCorrelación con target de regresión (justificación de exclusión):')
print(corr_leak.round(3))

Variables excluidas en regresión (leakage):
 - asignaturas_2sem_convalidadas
 - asignaturas_2sem_matriculadas
 - asignaturas_2sem_evaluadas
 - asignaturas_2sem_aprobadas
 - nota_media_2sem
 - asignaturas_2sem_sin_evaluacion
 - objetivo

Correlación con target de regresión (justificación de exclusión):
asignaturas_2sem_convalidadas      0.130
asignaturas_2sem_matriculadas      0.395
asignaturas_2sem_evaluadas         0.453
asignaturas_2sem_aprobadas         0.761
asignaturas_2sem_sin_evaluacion   -0.079
Name: nota_media_2sem, dtype: float64


> Las asignaturas aprobadas y evaluadas tienen suficiente correlación con el target como para justificar excluirlas. Para los otros tres casos, es debatible ya que no tienen tanta correlación, pero se excluyen igualmente ya que son datos pertinentes al segundo semestre y no estarían disponibles a la hora de predecir.

### Split train/test y preprocesado - Clasificación
Split 80/20 manteniendo la proporción de clases en ambos conjuntos, especialmente importante dado el ligero desbalanceo detectado en el EDA.
Las variables numéricas se normalizan con StandardScaler y las categóricas se transforman con OneHotEncoder. Se hace todo junto con un ColumnTransformer. Esto se hace para que unas variables no tengan importancia sobre otras solo por ser más grandes en magnitud.

In [6]:
X = df.drop(columns=['objetivo'])
y = df['objetivo']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

print(f'Train: {X_train_clf.shape[0]} instancias\nTest: {X_test_clf.shape[0]} instancias')

print('\nDistribución de \'objetivo\' en train:')
print(y_train_clf.value_counts(normalize=True).round(3))

print('\nDistribución de \'objetivo\' en test:')
print(y_test_clf.value_counts(normalize=True).round(3))

Train: 3539 instancias
Test: 885 instancias

Distribución de 'objetivo' en train:
objetivo
graduado       0.499
abandono       0.321
matriculado    0.179
Name: proportion, dtype: float64

Distribución de 'objetivo' en test:
objetivo
graduado       0.499
abandono       0.321
matriculado    0.180
Name: proportion, dtype: float64


In [7]:
num_cols_clf = X_train_clf.select_dtypes(include='number').columns.tolist()
cat_cols_clf = [c for c in X_train_clf.columns if c not in num_cols]

print(f'Variables numéricas: {len(num_cols_clf)}')
print(f'Variables categóricas: {len(cat_cols_clf)} (sin target)')

preprocessor_clf = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols_clf),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_clf),
    ], remainder="drop")

X_train_clf_t = preprocessor_clf.fit_transform(X_train_clf)
X_test_clf_t  = preprocessor_clf.transform(X_test_clf)

# Encode target
le = LabelEncoder()
y_train_clf_enc = le.fit_transform(y_train_clf)
y_test_clf_enc = le.transform(y_test_clf)

print(f'Clases: {le.classes_}')
print(f'Shape tras preprocesado: {X_train_clf_t.shape}')

Variables numéricas: 19
Variables categóricas: 17 (sin target)
Clases: ['abandono' 'graduado' 'matriculado']
Shape tras preprocesado: (3539, 241)


### Split train/test y preprocesado - Regresión
Se excluyen las variables de leakage definidas en la celda de leakage anterior y también la columna `objetivo` (clasificación), que no es una característica de entrada.

In [8]:
leak_cols.append('objetivo')
X = df.drop(columns=leak_cols)
y = df['nota_media_2sem']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y, test_size=0.2, random_state=0)

num_cols_reg = X_train_reg.select_dtypes(include='number').columns.tolist()
cat_cols_reg = [c for c in X_train_reg.columns if c not in num_cols]

preprocessor_reg = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols_reg),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_reg),
    ], remainder="drop")

X_train_reg_t = preprocessor_reg.fit_transform(X_train_reg)
X_test_reg_t = preprocessor_reg.transform(X_test_reg)

print(f'Train regresión: {X_train_reg_t.shape}  |  Test: {X_test_reg_t.shape}')
print(f'Target (primeras 5): {y_train_reg.values[:5]}')

Train regresión: (3539, 240)  |  Test: (885, 240)
Target (primeras 5): [12.5        11.77777778  0.         11.55       12.375     ]


### Guardar preprocesadores
Guardado de los preprocesadores para no reejecutar este notebook cada vez que se entrene un modelo.

In [9]:
joblib.dump(preprocessor_clf, '../preprocessors/preprocessor_clf.pkl')
joblib.dump(preprocessor_reg, '../preprocessors/preprocessor_reg.pkl')
joblib.dump(le, '../preprocessors/label_encoder_target.pkl')

['../preprocessors/label_encoder_target.pkl']